In [ ]:
import os
print("Working folder:", os.getcwd())
for name in os.listdir("."):
    print(name)

In [ ]:
import pandas as pd

files = ["nineteenFeaturesDf.csv", "dec12_18features.csv"]

for f in files:
    head = pd.read_csv(f, nrows=5)
    n_rows = sum(len(c) for c in pd.read_csv(f, usecols=[0], chunksize=500_000))
    print("=====", f)
    print("rows:", n_rows)
    print("columns:", len(head.columns))
    print(list(head.columns))
    display(head.head(3))

In [ ]:
import pandas as pd

df = pd.read_csv(
    "nineteenFeaturesDf.csv",
    dtype={"eventVersion": "string", "requestID": "string"},
)

df["eventTime"] = pd.to_datetime(df["eventTime"], utc=True, errors="coerce")

print("Shape:", df.shape)
print("Bad timestamps:", df["eventTime"].isna().sum())
print("Time range:", df["eventTime"].min(), "to", df["eventTime"].max())

print("\nDuplicate eventIDs:", df.duplicated(subset="eventID").sum())
print("Fully duplicated rows:", df.duplicated().sum())

print("\nMissing values per column:")
print(df.isna().sum())

In [ ]:
import pandas as pd

dtypes = {
    "eventVersion": "string",
    "requestID": "string",
    "userIdentityaccountId": "string",
    "requestParametersinstanceType": "string",
}
df = pd.read_csv("nineteenFeaturesDf.csv", dtype=dtypes)
df["eventTime"] = pd.to_datetime(df["eventTime"], utc=True)

before = len(df)

df = df.drop_duplicates(subset="eventID", keep="first")
print(f"Removed {before - len(df):,} duplicates -> {len(df):,} rows left")

df = df.sort_values("eventTime").reset_index(drop=True)

df = df.drop(columns=["userIdentityaccountId"])

df["has_error"] = df["errorCode"].notna().astype("int8")

fill_cols = [
    "userAgent", "userIdentitytype", "requestID",
    "userIdentityprincipalId", "userIdentityarn",
    "userIdentityaccessKeyId", "userIdentityuserName",
    "errorCode", "errorMessage", "requestParametersinstanceType",
]
df[fill_cols] = df[fill_cols].fillna("None")

print("\nRemaining missing values:", int(df.isna().sum().sum()))
print("Shape:", df.shape)
print("\nIdentity types:")
print(df["userIdentitytype"].value_counts())

df.to_parquet("clean_cloudtrail.parquet", index=False)
print("\nSaved clean_cloudtrail.parquet")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("clean_cloudtrail.parquet")

df["hour_of_day"] = df["eventTime"].dt.hour.astype("int8")
df["day_of_week"] = df["eventTime"].dt.dayofweek.astype("int8")

hour_bucket = df["eventTime"].dt.floor("h")
df["events_per_ip_1h"] = (
    df.groupby(["sourceIPAddress", hour_bucket])["eventID"].transform("size").astype("int32")
)
df["events_per_user_1h"] = (
    df.groupby(["userIdentityarn", hour_bucket])["eventID"].transform("size").astype("int32")
)

for col in ["sourceIPAddress", "eventName", "eventSource", "userAgent", "userIdentityarn"]:
    freq = df[col].value_counts(normalize=True)
    df[f"{col}_freq"] = df[col].map(freq).astype("float32")

df["is_new_ip_for_user"] = (
    df.groupby(["userIdentityarn", "sourceIPAddress"]).cumcount() == 0
).astype("int8")

id_dummies = pd.get_dummies(df["userIdentitytype"], prefix="idtype", dtype="int8")
df = pd.concat([df, id_dummies], axis=1)

feature_cols = (
    ["hour_of_day", "day_of_week", "events_per_ip_1h", "events_per_user_1h",
     "has_error", "is_new_ip_for_user"]
    + [c for c in df.columns if c.endswith("_freq")]
    + list(id_dummies.columns)
)

print("Number of features:", len(feature_cols))
print(feature_cols)
display(df[feature_cols].describe().T)

df.to_parquet("features_cloudtrail.parquet", index=False)
print("Saved features_cloudtrail.parquet")

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_parquet("features_cloudtrail.parquet")

feature_cols = (
    ["hour_of_day", "day_of_week", "events_per_ip_1h", "events_per_user_1h",
     "has_error", "is_new_ip_for_user"]
    + [c for c in df.columns if c.endswith("_freq") or c.startswith("idtype_")]
)
X = df[feature_cols]
print("Training on", X.shape, "with features:", feature_cols)

model = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1,
)
model.fit(X)

df["anomaly_score"] = -model.score_samples(X)
df["is_anomaly"] = (model.predict(X) == -1).astype("int8")

print("\nFlagged anomalies:", int(df["is_anomaly"].sum()), "of", len(df))

show_cols = ["eventTime", "userIdentitytype", "userIdentityarn", "sourceIPAddress",
             "eventName", "eventSource", "errorCode", "anomaly_score"]
top = df.sort_values("anomaly_score", ascending=False).head(100)
display(top[show_cols].head(25))

print("\nTop event names among anomalies:")
print(df.loc[df.is_anomaly == 1, "eventName"].value_counts().head(15))
print("\nIdentity types among anomalies:")
print(df.loc[df.is_anomaly == 1, "userIdentitytype"].value_counts())

df.to_parquet("scored_cloudtrail.parquet", index=False)
print("\nSaved scored_cloudtrail.parquet")

Ground Truth Labels

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")

print("Top 25 event names:")
print(df["eventName"].value_counts().head(25), "\n")

print("Top 15 error codes:")
print(df.loc[df["has_error"] == 1, "errorCode"].value_counts().head(15), "\n")

print("Top 15 user names:")
print(df["userIdentityuserName"].value_counts().head(15), "\n")

print("Role/user ARNs for non-IAMUser identities (top 15):")
print(df.loc[df["userIdentitytype"] != "IAMUser", "userIdentityarn"].value_counts().head(15))

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")

lines = []
def add(title, series):
    lines.append(f"\n=== {title} ===\n{series.to_string()}")

add("RunInstances by identity type",
    df.loc[df["eventName"] == "RunInstances", "userIdentitytype"].value_counts())
add("RunInstances by error code (top 10)",
    df.loc[df["eventName"] == "RunInstances", "errorCode"].value_counts().head(10))
add("RunInstances: top 10 user names",
    df.loc[df["eventName"] == "RunInstances", "userIdentityuserName"].value_counts().head(10))
add("Top 15 error codes (all events)",
    df.loc[df["has_error"] == 1, "errorCode"].value_counts().head(15))
add("Top 15 user names",
    df["userIdentityuserName"].value_counts().head(15))
add("Top 20 ARNs for non-IAMUser identities",
    df.loc[df["userIdentitytype"] != "IAMUser", "userIdentityarn"].value_counts().head(20))
add("AssumeRole by identity type",
    df.loc[df["eventName"] == "AssumeRole", "userIdentitytype"].value_counts())

with open("label_exploration.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("\n".join(lines[:4]))   # just the RunInstances parts
print("\nFull report saved to label_exploration.txt")

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")

for name in ["Level6", "backup"]:
    s = df[df["userIdentityuserName"] == name]
    print(f"\n===== {name} =====")
    print("Total events:", len(s))
    print("First/last event:", s["eventTime"].min(), "to", s["eventTime"].max())
    print("Distinct IPs:", s["sourceIPAddress"].nunique())
    print("Identity types:\n", s["userIdentitytype"].value_counts().to_string())
    print("Top 8 event names:\n", s["eventName"].value_counts().head(8).to_string())
    print("Error rate:", round(s["has_error"].mean(), 3))
    print("Top 3 user agents:\n", s["userAgent"].value_counts().head(3).to_string())

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")

s = df[df["userIdentityuserName"] == "backup"]
print("===== backup =====")
print("Total events:", len(s))
print("First/last event:", s["eventTime"].min(), "to", s["eventTime"].max())
print("Distinct IPs:", s["sourceIPAddress"].nunique())
print("Identity types:\n", s["userIdentitytype"].value_counts().to_string())
print("Top 8 event names:\n", s["eventName"].value_counts().head(8).to_string())
print("Error rate:", round(s["has_error"].mean(), 3))

# How many Level6 RunInstances share a user agent with backup?
l6 = df[(df["userIdentityuserName"] == "Level6") & (df["eventName"] == "RunInstances")]
shared = set(l6["userAgent"]) & set(s["userAgent"])
print("\nUser agents seen in both Level6 RunInstances and backup:", len(shared))

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")

users = {"Level6", "backup"}
is_user = df["userIdentityuserName"].isin(users)
name = df["eventName"]

df["label_category"] = "unlabelled"
df["label_source"] = "none"

mass  = is_user & (name == "RunInstances")
assume = is_user & (name == "AssumeRole")
enum  = is_user & name.str.match(r"^(Describe|List|Get)", na=False) & ~mass & ~assume

df.loc[mass,   ["label_category", "label_source"]] = ["mass_launch_attempts", "heuristic"]
df.loc[assume, ["label_category", "label_source"]] = ["assume_role", "heuristic"]
df.loc[enum,   ["label_category", "label_source"]] = ["enumeration", "heuristic"]

print(df["label_category"].value_counts().to_string())
print("\nShare labelled:", round((df["label_category"] != "unlabelled").mean(), 3))

# Early backup events, to check for owner activity
early = df[(df["userIdentityuserName"] == "backup")].sort_values("eventTime").head(30)
print("\nFirst 30 backup events:")
print(early[["eventTime", "sourceIPAddress", "eventName", "errorCode"]].to_string())

df.to_parquet("labelled_v0.parquet")

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")
df["label_category"] = "unlabelled"
df["label_source"] = "none"

def tag(mask, cat):
    m = mask & (df["label_category"] == "unlabelled")
    df.loc[m, ["label_category", "label_source"]] = [cat, "heuristic"]

arn = df["userIdentityarn"].astype(str)
user = df["userIdentityuserName"].astype(str)
is_lb = user.isin(["Level6", "backup"])

tag(df["userIdentitytype"] == "AWSService", "benign_service")
tag(arn.str.contains("SecurityMonkey|Cloudsploit|config-role|SummitRouteAudit|AWSServiceRole")
    | (user == "SecurityMokey"), "benign_audit_tooling")
tag(arn.str.contains("i-aa2d3b42e5c6e801a"), "credential_abuse")
tag(is_lb & (df["eventName"] == "RunInstances"), "mass_launch_attempts")
tag(is_lb, "pending_review")

print(df["label_category"].value_counts().to_string())
print("Share with a firm label:",
      round((~df["label_category"].isin(["unlabelled", "pending_review"])).mean(), 3))

ri = df[df["eventName"] == "RunInstances"]
print("\nRunInstances with no error:", int((ri["has_error"] == 0).sum()))

owner_ip = "255.253.125.115"
o = df[df["sourceIPAddress"] == owner_ip]
print("\nOwner-IP events:", len(o))
print(o["eventTime"].dt.year.value_counts().sort_index().to_string())
print(o["userIdentityuserName"].value_counts().head(5).to_string())

root = df[df["userIdentitytype"] == "Root"]
print("\nRoot events:", len(root), "| distinct IPs:", root["sourceIPAddress"].nunique())
print("Share of Root from owner IP:", round((root["sourceIPAddress"] == owner_ip).mean(), 3))
print(root["sourceIPAddress"].value_counts().head(5).to_string())

h = df[user == "HIDDEN_DUE_TO_SECURITY_REASONS"]
print("\nHIDDEN user events by eventName:\n", h["eventName"].value_counts().to_string())

df.to_parquet("labelled_v1.parquet")

In [ ]:
import pandas as pd

df = pd.read_parquet("clean_cloudtrail.parquet")
df["label_category"] = "unlabelled"
df["label_source"] = "none"

def tag(mask, cat):
    m = mask & (df["label_category"] == "unlabelled")
    df.loc[m, ["label_category", "label_source"]] = [cat, "heuristic"]

arn  = df["userIdentityarn"].astype(str)
user = df["userIdentityuserName"].astype(str)
ip   = df["sourceIPAddress"].astype(str)
name = df["eventName"].astype(str)
is_lb = user.isin(["Level6", "backup"])
denied = df["errorCode"].astype(str).str.contains("AccessDenied|UnauthorizedOperation")
owner_ip = "255.253.125.115"

tag(df["userIdentitytype"] == "AWSService", "benign_service")
tag(arn.str.contains("SecurityMonkey|Cloudsploit|config-role|SummitRouteAudit|AWSServiceRole")
    | (user == "SecurityMokey"), "benign_audit_tooling")
tag(ip == owner_ip, "benign_owner_activity")
tag(ip == "support.amazonaws.com", "benign_service")
tag(arn.str.contains("i-aa2d3b42e5c6e801a"), "credential_abuse")
tag(is_lb & (name == "RunInstances"), "mass_launch_attempts")
tag(user == "HIDDEN_DUE_TO_SECURITY_REASONS", "failed_console_login")
tag(is_lb & denied, "access_denied_probing")
tag(is_lb & (df["has_error"] == 0), "shared_cred_success")
tag(is_lb, "shared_cred_other_errors")

print(df["label_category"].value_counts().to_string())
print("Total rows:", len(df))

df.to_parquet("labelled_v2.parquet")
print("Saved labelled_v2.parquet")

In [ ]:
import pandas as pd

df = pd.read_parquet("labelled_v2.parquet")
sc = pd.read_parquet("scored_cloudtrail.parquet", columns=["eventTime", "is_anomaly"])

same_order = len(sc) == len(df) and (sc["eventTime"].values == df["eventTime"].values).all()
print("Row order matches:", same_order)

if same_order:
    df["is_anomaly"] = sc["is_anomaly"].values
    print("Flagged:", int((df["is_anomaly"] == 1).sum()), "(should be 12256)\n")

    flagged = df[df["is_anomaly"] == 1]
    summary = pd.DataFrame({
        "flagged": flagged["label_category"].value_counts(),
        "total_in_category": df["label_category"].value_counts(),
    }).fillna(0).astype(int)
    summary["pct_of_category_flagged"] = (100 * summary["flagged"] / summary["total_in_category"]).round(2)
    print(summary.sort_values("flagged", ascending=False).to_string())

In [ ]:
import pandas as pd

df = pd.read_parquet("labelled_v2.parquet")
sc = pd.read_parquet("scored_cloudtrail.parquet", columns=["is_anomaly"])
df["is_anomaly"] = sc["is_anomaly"].values

u = df[(df["is_anomaly"] == 1) & (df["label_category"] == "unlabelled")]
print("Flagged and unlabelled:", len(u))
print("\nBy identity type:\n", u["userIdentitytype"].value_counts().to_string())
print("\nTop 10 event names:\n", u["eventName"].value_counts().head(10).to_string())
print("\nTop 8 identities:\n", u["userIdentityarn"].astype(str).str[:70].value_counts().head(8).to_string())
print("\nTop 5 source IPs:\n", u["sourceIPAddress"].value_counts().head(5).to_string())
print("\nError rate:", round(u["has_error"].mean(), 3))
print("\nBy year:\n", u["eventTime"].dt.year.value_counts().sort_index().to_string())

In [ ]:
import pandas as pd

df = pd.read_parquet("labelled_v2.parquet")
sc = pd.read_parquet("scored_cloudtrail.parquet", columns=["is_anomaly", "anomaly_score"])
df["is_anomaly"] = sc["is_anomaly"].values
df["anomaly_score"] = sc["anomaly_score"].values

print("Label categories (all rows):")
print(df["label_category"].value_counts().to_string())

print("\nFlagged vs not flagged, by label category:")
print(pd.crosstab(df["label_category"], df["is_anomaly"], margins=True).to_string())

print("\nShare of each category the model flagged (%):")
print((df.groupby("label_category")["is_anomaly"].mean() * 100).round(2).to_string())

In [ ]:
import pandas as pd

df = pd.read_parquet("labelled_v2.parquet")
sc = pd.read_parquet("scored_cloudtrail.parquet", columns=["is_anomaly", "anomaly_score"])
df["is_anomaly"] = sc["is_anomaly"].values
df["anomaly_score"] = sc["anomaly_score"].values

flag = df[df["is_anomaly"] == 1].copy()
flag["eventTime"] = pd.to_datetime(flag["eventTime"], errors="coerce")  # safety: makes sure time is a real date
flag["hour"] = flag["eventTime"].dt.floor("h")

inc = flag.groupby(["userIdentityarn", "sourceIPAddress", "hour"], sort=False).agg(
    n_events=("eventName", "size"),
    n_distinct_events=("eventName", "nunique"),
    top_event=("eventName", lambda s: s.value_counts().index[0]),
    error_rate=("has_error", "mean"),
    max_score=("anomaly_score", "max"),
    label_category=("label_category", lambda s: s.value_counts().index[0]),
    identity_type=("userIdentitytype", "first"),
).reset_index()

print("Flagged events:", len(flag), "-> incidents:", len(inc))
print("\nIncidents by label category:\n", inc["label_category"].value_counts().to_string())
print("\nEvents per incident:\n", inc["n_events"].describe().round(1).to_string())
inc.to_parquet("incidents.parquet", index=False)
print("\nSaved incidents.parquet")

In [ ]:
u = inc[inc["label_category"] == "unlabelled"]
print("Top identities in unlabelled incidents:\n", u["userIdentityarn"].value_counts().head(5).to_string())
print("\nTop events in unlabelled incidents:\n", u["top_event"].value_counts().head(5).to_string())

print("\n10 biggest incidents:")
print(inc.sort_values("n_events", ascending=False).head(10)[
    ["userIdentityarn", "sourceIPAddress", "hour", "n_events", "top_event", "label_category"]
].to_string(index=False))

In [ ]:
noise = (
    (inc["label_category"] == "unlabelled")
    & (inc["userIdentityarn"] == "arn:aws:sts::811596193553:assumed-role/Level6/Level6")
    & (inc["top_event"] == "CreateLogStream")
)
inc["is_routine_noise"] = noise
print("Routine Lambda logging incidents:", noise.sum())
print("Incidents left for analysis:", (~noise).sum())
print(inc.loc[~noise, "label_category"].value_counts().to_string())
inc.to_parquet("incidents_v2.parquet", index=False)
print("Saved incidents_v2.parquet")

In [ ]:
KEY_COLS = [c for c in ["eventName", "eventSource", "sourceIPAddress", "userIdentityarn", "errorCode"] if c in df.columns]

def collapse(rows):
    """Merge back-to-back identical events into one row with a count."""
    r = rows[SHOW_COLS].copy()
    if "userAgent" in r.columns:
        r["userAgent"] = r["userAgent"].astype(str).str[:40]
    keys = r[KEY_COLS].astype(str)
    block = (keys != keys.shift()).any(axis=1).cumsum()
    agg = {c: (c, "first") for c in SHOW_COLS}
    agg["count"] = ("eventName", "size")
    return r.groupby(block).agg(**agg).reset_index(drop=True)

def head_tail(t, n):
    return t if len(t) <= n else pd.concat([t.head(n // 2), t.tail(n - n // 2)])

def list_counts(s):
    return ", ".join(f"{k} x{v}" for k, v in s.items()) or "none"

def get_context(incident, df, window_hours=1, core_rows=20, around_rows=10):
    h, arn, ip = incident["hour"], incident["userIdentityarn"], incident["sourceIPAddress"]
    start, end = h - pd.Timedelta(hours=window_hours), h + pd.Timedelta(hours=1 + window_hours)
    in_time = (df["eventTime"] >= start) & (df["eventTime"] < end)
    ctx = df.loc[in_time & ((df["userIdentityarn"] == arn) | (df["sourceIPAddress"] == ip))].sort_values("eventTime")

    in_hour = (ctx["eventTime"] >= h) & (ctx["eventTime"] < h + pd.Timedelta(hours=1))
    core = ctx[in_hour & (ctx["userIdentityarn"] == arn) & (ctx["sourceIPAddress"] == ip)]
    around = ctx.drop(core.index)

    other_ids = ctx.loc[ctx["sourceIPAddress"] == ip, "userIdentityarn"].value_counts().drop(arn, errors="ignore").head(5)
    other_ips = ctx.loc[ctx["userIdentityarn"] == arn, "sourceIPAddress"].value_counts().drop(ip, errors="ignore").head(5)

    lines = [
        f"INCIDENT: identity={arn} | ip={ip} | hour={h} | identity type={incident['identity_type']} | flagged events={incident['n_events']}",
        f"CONTEXT WINDOW: {start} to {end} | related events={len(ctx)}",
        "TOP EVENTS IN WINDOW: " + list_counts(ctx["eventName"].value_counts().head(10)),
        "OTHER IDENTITIES USING THIS IP: " + list_counts(other_ids),
        "OTHER IPs USED BY THIS IDENTITY: " + list_counts(other_ips),
    ]
    if "errorCode" in ctx.columns:
        lines.append("ERRORS: " + list_counts(ctx.loc[ctx["errorCode"] != "None", "errorCode"].value_counts().head(5)))

    if len(core):
        core_c = head_tail(collapse(core), core_rows)
        lines.append(f"\nINCIDENT-HOUR EVENTS ({len(core)} events, {len(core_c)} rows shown, repeats merged):")
        lines.append(core_c.to_string(index=False))
    if len(around):
        ar_c = head_tail(collapse(around), around_rows)
        lines.append(f"\nSURROUNDING EVENTS ({len(around)} events, {len(ar_c)} rows shown, repeats merged):")
        lines.append(ar_c.to_string(index=False))
    return "\n".join(lines)

for ex in [ex1, ex2]:
    print(get_context(ex, df))
    print("\n" + "=" * 100 + "\n")